# 5. Salary Model Training

**Objective:** train a model that predicts `salary_midpoint_lpa` from job attributes (experience, role, city, company size, work mode) so `ui/salary.py` can give a user an estimated salary range instead of the placeholder page.

**Caveat going in:** only ~12% of postings in `careerlens_cleaned.csv` have a disclosed salary (2,768 of 23,201). We train and evaluate on that subset only — everything below is scoped to those rows.

In [1]:
import pandas as pd
import numpy as np
import json
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../data/processed/careerlens_cleaned.csv", keep_default_na=False)
df.shape

(23201, 33)

In [2]:
disclosed = df[df["salary_disclosed"].astype(str).isin(["True", "true", "1"])].copy()

print(f"{len(disclosed):,} of {len(df):,} postings have a disclosed salary "
      f"({len(disclosed)/len(df):.1%})")
disclosed["salary_midpoint_lpa"].describe()

2,768 of 23,201 postings have a disclosed salary (11.9%)


count    2768.000000
mean       15.320972
std        10.302711
min         0.000000
25%         7.500000
50%        14.000000
75%        21.500000
max        87.500000
Name: salary_midpoint_lpa, dtype: float64

In [3]:
top_cities = disclosed["primary_city"].value_counts().head(15).index
disclosed["city_bucketed"] = disclosed["primary_city"].where(
    disclosed["primary_city"].isin(top_cities), "Other"
)

NUMERIC = ["experience_min_yrs", "experience_max_yrs", "company_rating", "skills_count"]
CATEGORICAL = ["role_category", "skill_domain", "work_mode", "company_size_bucket", "city_bucketed"]
TARGET = "salary_midpoint_lpa"

X = disclosed[NUMERIC + CATEGORICAL]
y = disclosed[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

((2214, 9), (554, 9))

In [4]:
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
], remainder="passthrough")

candidates = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=4, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42),
}

results = {}
for name, model in candidates.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r2 = r2_score(y_test, pred)
    cv = cross_val_score(pipe, X, y, cv=5, scoring="r2")

    results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2, "CV_R2_mean": cv.mean(), "CV_R2_std": cv.std()}
    print(f"{name:20s} MAE={mae:6.2f}  RMSE={rmse:6.2f}  R2={r2:.3f}  CV_R2={cv.mean():.3f} (+/- {cv.std():.3f})")

best_name = max(results, key=lambda k: results[k]["R2"])
print("\nBest model on held-out R2:", best_name)

LinearRegression     MAE=  5.52  RMSE=  7.59  R2=0.368  CV_R2=0.315 (+/- 0.107)
RandomForest         MAE=  5.32  RMSE=  7.58  R2=0.370  CV_R2=0.402 (+/- 0.090)
GradientBoosting     MAE=  5.35  RMSE=  7.50  R2=0.384  CV_R2=0.416 (+/- 0.088)

Best model on held-out R2: GradientBoosting


**GradientBoostingRegressor** wins, though only narrowly over RandomForest — none of the three is dramatically better, which tells us the ceiling here is mostly about the *features*, not the *model*. R² around 0.38 means these features (experience, role, city, company size, work mode) explain roughly 38% of salary variance. The rest — negotiation, individual performance, unlisted benefits — isn't in this dataset, so we surface a **range**, not a single confident number, in the app.

## 5.4 Train the final model on all disclosed data

In [5]:
final_model = GradientBoostingRegressor(
    n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42
)
final_pipe = Pipeline([("prep", preprocess), ("model", final_model)])
final_pipe.fit(X, y)

import os
os.makedirs("../models", exist_ok=True)
joblib.dump(final_pipe, "../models/salary_model.pkl")
print("Saved ../models/salary_model.pkl")

Saved ../models/salary_model.pkl


## 5.5 Save metadata for the Streamlit form

`ui/salary.py` needs to know the valid dropdown values and slider ranges without hardcoding them — this JSON is the single source of truth, generated straight from the training data used to fit the model above.

In [6]:
meta = {
    "numeric_features": NUMERIC,
    "categorical_features": CATEGORICAL,
    "target": TARGET,
    "cities": sorted(top_cities.tolist()) + ["Other"],
    "role_categories": sorted(disclosed["role_category"].unique().tolist()),
    "skill_domains": sorted(disclosed["skill_domain"].unique().tolist()),
    "work_modes": sorted(disclosed["work_mode"].unique().tolist()),
    "company_size_buckets": sorted(disclosed["company_size_bucket"].unique().tolist()),
    "experience_min_range": [float(disclosed["experience_min_yrs"].min()), float(disclosed["experience_min_yrs"].max())],
    "experience_max_range": [float(disclosed["experience_max_yrs"].min()), float(disclosed["experience_max_yrs"].max())],
    "company_rating_range": [float(disclosed["company_rating"].min()), float(disclosed["company_rating"].max())],
    "skills_count_range": [int(disclosed["skills_count"].min()), int(disclosed["skills_count"].max())],
    "training_rows": int(len(disclosed)),
    "model_type": "GradientBoostingRegressor",
}

# residual std from the held-out test predictions above -> used to show a +/- range,
# not just a single point estimate, given the modest R2
test_pred = Pipeline([("prep", preprocess), ("model", candidates["GradientBoosting"])]).fit(X_train, y_train).predict(X_test)
residual_std = float(np.std(y_test - test_pred))

meta["metrics"] = {
    "MAE_lpa": round(results["GradientBoosting"]["MAE"], 2),
    "RMSE_lpa": round(results["GradientBoosting"]["RMSE"], 2),
    "R2": round(results["GradientBoosting"]["R2"], 3),
    "residual_std_lpa": round(residual_std, 2),
}

with open("../data/processed/salary_model_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

meta

{'numeric_features': ['experience_min_yrs',
  'experience_max_yrs',
  'company_rating',
  'skills_count'],
 'categorical_features': ['role_category',
  'skill_domain',
  'work_mode',
  'company_size_bucket',
  'city_bucketed'],
 'target': 'salary_midpoint_lpa',
 'cities': ['Ahmedabad',
  'Bangalore',
  'Chennai',
  'Delhi',
  'Gurgaon',
  'Hyderabad',
  'Kolkata',
  'Mumbai',
  'Mumbai Suburban',
  'Mumbai(Airoli)',
  'Noida',
  'Pune',
  'Pune(Baner)',
  'Pune(Hinjewadi Phase 1)',
  'Remote',
  'Other'],
 'role_categories': ['Business Analyst',
  'Data Analyst',
  'Data Engineer',
  'Data Scientist',
  'Machine Learning Engineer',
  'Python Developer'],
 'skill_domains': ['AI/ML/DL',
  'Business Intelligence',
  'Cloud & DevOps',
  'Data Engineering',
  'Data Science'],
 'work_modes': ['Hybrid', 'On-site', 'Remote'],
 'company_size_buckets': ['Large (1000+)',
  'Mid (100-999)',
  'Small/Startup (<100)'],
 'experience_min_range': [0.0, 20.0],
 'experience_max_range': [0.0, 31.0],
 'com

## Conclusion

- Trained on 2,768 postings with a disclosed salary (12% of the full dataset); features: experience, role category, skill domain, work mode, company size/rating, top-15 city.
- Final model: `GradientBoostingRegressor`, test R² ≈ 0.38, MAE ≈ ₹5.3 LPA.
- Modest R² is expected given the missing signal (negotiation, seniority within a role, benefits) — the app shows a **predicted range** (point estimate ± residual std), not a false-precision single number.
- Outputs: `models/salary_model.pkl` (the fitted pipeline) and `data/processed/salary_model_meta.json` (dropdown options + ranges + metrics), both consumed directly by `ui/salary.py`.

**Next stage:** wire this into `ui/salary.py`.